# Evaluación de los modelos OBB en el dataset externo

Evalúa **todos los modelos de `models/`** (los mismos que ve la app, con el mismo pre y
post-proceso) sobre el dataset externo `external_dataset/billetesprueba 2.yolov8-obb` (143 fotos de móvil,
208 billetes).

| sección | qué responde |
|---|---|
| 1. Métricas principales | precision, recall, F1, mAP50, mAP75, mAP50-95 y la confianza (media, mediana, desviación) |
| 2. Curvas PR y umbral de confianza | cómo cambian P / R / F1 con el umbral y cuál sería el óptimo de cada modelo |
| 3. Confianza | si la confianza separa los aciertos de los falsos positivos |
| 4. Calidad de la caja | IoU y error de ángulo de los aciertos, recall por tamaño del billete |
| 5. Estudio de NMS | sensibilidad al umbral IoU del NMS |
| 6. Estudio de rotación | robustez al girar la foto |
| 7. Estudio de resolución | robustez a fotos de menor resolución |
| 8. Velocidad y ejemplos | latencia y las fotos donde cada modelo falla |

### Cómo usarlo

1. Pon los modelos en `models/` (igual que para la app) y el dataset en `external_dataset/`
   (ver los README de esas carpetas).
2. Abre el notebook con el entorno del proyecto: `uv run jupyter lab evaluacion_modelos.ipynb`,
   o en VS Code eligiendo como kernel el Python de `.venv`.
3. Revisa la celda de **configuración** (la primera de código) y ejecuta todo
   (*Run → Run All Cells*).

| variable | qué hace |
|---|---|
| `DATASET` | carpeta del dataset (YOLOv8-OBB) |
| `MODELS_DIR` | carpeta de modelos; se evalúan todos los que haya |
| `AP_SCORE_THR` | confianza mínima de las detecciones que entran en el mAP (0.05) |
| `OPERATING_THR` | umbral de trabajo para precision / recall / F1 y la confianza (0.5) |
| `NMS_IOU` | IoU del NMS, igual para todos los modelos (0.3) |
| `NMS_STUDY`, `ROT_ANGLES`, `RES_SIZES` | valores de los estudios de NMS, rotación y resolución |
| `RUN_STUDIES` | `False` = solo métricas principales (~2 min en vez de ~10 min con 2 modelos) |
| `QUICK` | `True` = estudios de rotación/resolución con 1 de cada 3 fotos |

La inferencia se hace una vez y se guarda en `eval_cache/`: volver a ejecutar es
instantáneo y cambiar `OPERATING_THR` no obliga a repetirla. Si cambian los modelos, el
dataset o los valores de los estudios, la caché se rehace sola. Borrar `eval_cache/` es
siempre seguro.

## ¿Qué umbrales usar?

Hay que separar dos tipos de métrica, porque piden umbrales distintos:

**mAP50 / mAP75 / mAP50-95 → umbral de confianza bajo (0.05), igual para todos.** El AP ya
recorre todos los umbrales (es el área bajo la curva precisión-recall), así que no depende
del punto de trabajo; si se cortara a 0.5 se estaría amputando la curva y castigando a los
modelos poco confiados. Es lo que hacen los `evaluate.py` de los tres repos (0.05) y lo que
permite comparar con sus READMEs.

**Precision / recall / F1 y estadísticas de confianza → un umbral de trabajo.** Aquí se
reportan dos cosas, y conviene mirar las dos:

1. **Umbral fijo común (0.5)** → la comparación principal. Es el valor de producción de los
   tres repos y el que usa la app, así que mide lo que verá el usuario. Es justo: nadie se
   ajusta al test.
2. **Umbral que maximiza el F1 de cada modelo** → el "techo" de cada modelo. Es útil porque
   las confianzas de arquitecturas distintas no están calibradas igual (YOLOX multiplica
   objectness × clase y suele dar scores más bajos; un umbral fijo puede castigarlo sin
   que detecte peor). **Pero es optimista**: el umbral se elige mirando este mismo
   dataset. Lo correcto es elegirlo en la partición de *validación* del entrenamiento y
   aplicarlo aquí; si ese umbral y el óptimo de aquí se parecen, el modelo es estable.

**NMS IoU → fijo e igual para todos (0.3)**, el valor con el que están las tablas de los
READMEs. En la sección 5 se ve cuánto cambia con otros valores (0.1 es el de DOTA/mmrotate,
suprime billetes apilados; 0.5 los respeta más).

Los TP se cuentan con IoU rotado ≥ 0.5 frente a la anotación. El AP es el de los repos
(mmrotate, VOC "area"), así que los números son comparables con sus READMEs.

In [ ]:
from pathlib import Path
import hashlib, pickle, time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from obbcompare.config import discover
from obbcompare.detector import OnnxDetector, postprocess, corners
from obbcompare.evaluation import (
    IOU_THRS, ImageResult, angle_error_deg, average_precision, downscale_for_study, load_image,
    load_yolo_obb, match, matched_pairs, poly_to_rbox, pr_curve, prf_at, prf_curve, resize_long_side,
    rotate, summarize,
)

# ------------------------------------------------------------------ configuración
DATASET = Path("external_dataset/billetesprueba 2.yolov8-obb")
MODELS_DIR = Path("models")
AP_SCORE_THR = 0.05       # detecciones que entran en el AP (como los evaluate.py de los repos)
OPERATING_THR = 0.5       # umbral de trabajo para P / R / F1 y la confianza
NMS_IOU = 0.3             # igual para todos los modelos
NMS_STUDY = [0.1, 0.2, 0.3, 0.5, 0.7]
ROT_ANGLES = [0, 15, 30, 45, 60, 75, 90, 135, 180]      # grados, antihorario
RES_SIZES = [320, 480, 640, 800, 1024, 1600]            # lado largo de la foto, px
RUN_STUDIES = True        # False: solo métricas principales (unos 2 min en vez de ~15)
QUICK = False             # True: estudios de rotación/resolución con 1 de cada 3 fotos

In [ ]:
# ------------------------------------------------------------------ estilo de las gráficas
# un color fijo por modelo (por orden en models/, nunca por ranking)
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#b5b4ad", "axes.grid": True, "grid.color": "#e4e3dc", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "lines.markersize": 6, "legend.frameon": False, "axes.titleweight": "bold",
    "axes.titlesize": 11, "axes.labelsize": 10, "font.size": 10,
})

## 0. Datos, modelos e inferencia

In [ ]:
samples = load_yolo_obb(DATASET)
configs, warnings = discover(MODELS_DIR)
for w in warnings:
    print("AVISO:", w)
NAMES = [c.name for c in configs]
COLORS = {n: PALETTE[i % len(PALETTE)] for i, n in enumerate(NAMES)}
print(f"{len(samples)} imágenes, {sum(len(s.polys) for s in samples)} billetes anotados")
for c in configs:
    print(f"  · {c.name}  [{c.family}]  {c.onnx_path.name}")

In [ ]:
def cache_key():
    h = hashlib.sha1()
    for c in configs:
        h.update(f"{c.onnx_path}:{c.onnx_path.stat().st_mtime_ns}:{c.onnx_path.stat().st_size}".encode())
    h.update(repr((AP_SCORE_THR, NMS_IOU, NMS_STUDY, ROT_ANGLES, RES_SIZES, RUN_STUDIES, QUICK,
                   [s.image_path.relative_to(DATASET).as_posix() for s in samples])).encode())
    return h.hexdigest()[:16]

CACHE = Path("eval_cache") / f"results_{cache_key()}.pkl"


def detect(det, img, polys, labels, nms_list=(NMS_IOU,)):
    raw = det.infer(img)
    out = {nms: ImageResult.from_detections(
        postprocess(raw, AP_SCORE_THR, nms, max_det=2000, nms_pre=2000), polys, labels) for nms in nms_list}
    return out, raw.ms


def run_all():
    detectors = {c.name: OnnxDetector(c) for c in configs}
    R = {"base": {n: [] for n in NAMES}, "nms": {k: {n: [] for n in NAMES} for k in NMS_STUDY},
         "rot": {a: {n: [] for n in NAMES} for a in ROT_ANGLES},
         "res": {s: {n: [] for n in NAMES} for s in RES_SIZES},
         "ms": {n: [] for n in NAMES}, "img_hw": []}
    t0 = time.time()
    for i, s in enumerate(samples):
        img = load_image(s.image_path)
        R["img_hw"].append(img.shape[:2])
        for n, det in detectors.items():
            res, ms = detect(det, img, s.polys, s.labels, sorted(set(NMS_STUDY) | {NMS_IOU}))
            R["base"][n].append(res[NMS_IOU])
            R["ms"][n].append(ms)
            for k in NMS_STUDY:
                R["nms"][k][n].append(res[k])
        if RUN_STUDIES and (not QUICK or i % 3 == 0):
            small, polys = downscale_for_study(img, s.polys)
            for a in ROT_ANGLES:
                im_r, p_r = rotate(small, polys, a)
                for n, det in detectors.items():
                    R["rot"][a][n].append(detect(det, im_r, p_r, s.labels)[0][NMS_IOU])
            for size in RES_SIZES:
                im_s, p_s = resize_long_side(img, s.polys, size)
                for n, det in detectors.items():
                    R["res"][size][n].append(detect(det, im_s, p_s, s.labels)[0][NMS_IOU])
        if (i + 1) % 20 == 0:
            print(f"{i + 1}/{len(samples)} imágenes · {time.time() - t0:.0f} s")
    return R


if CACHE.exists():
    R = pickle.loads(CACHE.read_bytes())
    print(f"resultados leídos de {CACHE}")
else:
    R = run_all()
    CACHE.parent.mkdir(exist_ok=True)
    CACHE.write_bytes(pickle.dumps(R))
    print(f"guardado en {CACHE}")

## 1. Métricas principales

Umbral de trabajo común (`OPERATING_THR`) para precision / recall / F1 / confianza; mAP con
todas las detecciones de score ≥ 0.05. La confianza (media, mediana, desviación) es la de
las predicciones que pasan el umbral de trabajo.

In [ ]:
def table(results_by_model, thr=OPERATING_THR):
    rows = {n: summarize(results_by_model[n], thr) for n in NAMES}
    return pd.DataFrame(rows).T

main = table(R["base"])
main["ms/imagen"] = [np.mean(R["ms"][n]) for n in NAMES]
fmt = {c: "{:.3f}" for c in ["precision", "recall", "f1", "mAP50", "mAP75", "mAP50-95",
                              "conf_mean", "conf_median", "conf_std"]}
fmt.update({"TP": "{:.0f}", "FP": "{:.0f}", "FN": "{:.0f}", "ms/imagen": "{:.0f}"})
metric_cols = ["precision", "recall", "f1", "mAP50", "mAP75", "mAP50-95"]
display(main.style.format(fmt).highlight_max(subset=metric_cols, color="#cde2fb", axis=0)
        .set_caption(f"Umbral de trabajo {OPERATING_THR} · NMS IoU {NMS_IOU} · azul = mejor"))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
w = 0.8 / len(NAMES)
x = np.arange(len(metric_cols))
for i, n in enumerate(NAMES):
    vals = main.loc[n, metric_cols].astype(float).values
    bars = ax.bar(x + (i - (len(NAMES) - 1) / 2) * w, vals, w * 0.92, color=COLORS[n], label=n)
    ax.bar_label(bars, fmt="%.2f", fontsize=7.5, padding=2, color="#52514e")
ax.set_xticks(x, metric_cols)
ax.set_ylim(0, 1.08)
ax.set_title(f"Métricas en el dataset externo (P/R/F1 con umbral {OPERATING_THR})")
ax.legend(loc="lower left", fontsize=8)
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

## 2. Curvas precisión-recall y elección del umbral de confianza

A la izquierda las curvas PR (el mAP es el área bajo ellas). A la derecha, para cada modelo,
cómo cambian precision, recall y F1 al mover el umbral de confianza: la línea vertical gris
es el umbral común y el punto marca el umbral que maximiza el F1.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, iou in zip(axes, [0.5, 0.75]):
    for n in NAMES:
        p, r, _, _ = pr_curve(R["base"][n], iou)
        ap = average_precision(R["base"][n], iou)
        ax.plot(r, p, color=COLORS[n], label=f"{n}  (AP {ap:.3f})")
    ax.set(xlabel="recall", ylabel="precision", title=f"Curva PR · IoU {iou}", xlim=(0, 1.01), ylim=(0, 1.02))
    ax.legend(loc="lower left", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
THRS = np.round(np.arange(0.05, 0.96, 0.01), 2)
curves = {n: prf_curve(R["base"][n], THRS) for n in NAMES}
fig, axes = plt.subplots(1, len(NAMES), figsize=(4.6 * len(NAMES), 3.8), sharey=True, squeeze=False)
best = {}
for ax, n in zip(axes[0], NAMES):
    c = curves[n]
    ax.plot(THRS, c[:, 0], color=COLORS[n], ls="--", lw=1.6, label="precision")
    ax.plot(THRS, c[:, 1], color=COLORS[n], ls=":", lw=1.8, label="recall")
    ax.plot(THRS, c[:, 2], color=COLORS[n], label="F1")
    k = int(np.argmax(c[:, 2]))
    best[n] = THRS[k]
    ax.axvline(OPERATING_THR, color="#8a8983", lw=1)
    ax.scatter([THRS[k]], [c[k, 2]], s=50, color=COLORS[n], edgecolor="#fcfcfb", linewidth=2, zorder=3)
    ax.annotate(f"F1 máx {c[k, 2]:.3f}\numbral {THRS[k]:.2f}", (THRS[k], c[k, 2]), xytext=(6, -30),
                textcoords="offset points", fontsize=8, color="#52514e")
    ax.set(title=n, xlabel="umbral de confianza", ylim=(0, 1.02))
    ax.legend(loc="lower left", fontsize=8)
axes[0][0].set_ylabel("valor")
plt.tight_layout(); plt.show()

opt = pd.DataFrame({n: summarize(R["base"][n], best[n]) for n in NAMES}).T
opt.insert(0, "umbral F1-máx", [best[n] for n in NAMES])
display(opt[["umbral F1-máx", "precision", "recall", "f1", "conf_mean", "conf_median", "conf_std", "TP", "FP", "FN"]]
        .style.format({**fmt, "umbral F1-máx": "{:.2f}"})
        .set_caption("Con el umbral que maximiza el F1 de cada modelo (optimista: elegido sobre este dataset)"))

## 3. ¿La confianza separa aciertos de errores?

Un buen detector da confianzas altas a los aciertos (TP) y bajas a los falsos positivos (FP):
cuanto menos se solapen los histogramas, más fácil es elegir un umbral. Se muestran todas
las detecciones con score ≥ 0.05, con el umbral común marcado.

In [ ]:
fig, axes = plt.subplots(1, len(NAMES), figsize=(4.6 * len(NAMES), 3.4), sharey=True, squeeze=False)
bins = np.linspace(0, 1, 26)
rows = {}
for ax, n in zip(axes[0], NAMES):
    tp_rows, fp_scores, _ = matched_pairs(R["base"][n], AP_SCORE_THR)
    tp_scores = np.array([r[2] for r in tp_rows])
    ax.hist(tp_scores, bins, color=COLORS[n], alpha=0.85, label=f"TP ({len(tp_scores)})")
    ax.hist(fp_scores, bins, color="#8a8983", alpha=0.7, label=f"FP ({len(fp_scores)})")
    ax.axvline(OPERATING_THR, color="#1a1a19", lw=1)
    ax.set(title=n, xlabel="confianza")
    ax.legend(fontsize=8)
    rows[n] = {"TP media": tp_scores.mean(), "TP mediana": np.median(tp_scores), "TP std": tp_scores.std(),
               "FP media": fp_scores.mean() if len(fp_scores) else np.nan,
               "FP mediana": np.median(fp_scores) if len(fp_scores) else np.nan,
               "FP con score ≥ umbral": int((fp_scores >= OPERATING_THR).sum())}
axes[0][0].set_ylabel("nº de detecciones")
plt.tight_layout(); plt.show()
display(pd.DataFrame(rows).T.style.format({**{c: "{:.3f}" for c in ["TP media", "TP mediana", "TP std", "FP media", "FP mediana"]},
                                            "FP con score ≥ umbral": "{:.0f}"}))

## 4. Calidad de la caja y tamaño del billete

- **IoU de los aciertos**: lo bien ajustada que está la caja (es lo que separa el mAP50 del
  mAP75 y el mAP50-95).
- **Error de ángulo**: diferencia entre la dirección del lado largo de la predicción y la de
  la anotación, en grados (0-90; un rectángulo girado 180° es el mismo).
- **Recall por tamaño**: los billetes se reparten en tres grupos iguales según su tamaño
  relativo en la foto (raíz del área del billete / área de la imagen).

In [ ]:
gt_rel = []  # tamaño relativo de cada billete anotado
for s, (h, w) in zip(samples, R["img_hw"]):
    rb = poly_to_rbox(s.polys)
    gt_rel.append(np.sqrt(rb[:, 2] * rb[:, 3] / (h * w)))
all_rel = np.concatenate(gt_rel)
edges = np.quantile(all_rel, [0, 1 / 3, 2 / 3, 1])
size_names = [f"pequeño\n(<{edges[1]:.2f})", f"mediano\n({edges[1]:.2f}-{edges[2]:.2f})", f"grande\n(>{edges[2]:.2f})"]
size_of = lambda v: min(int(np.searchsorted(edges[1:-1], v, side="right")), 2)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
recall_by_size, quality = {}, {}
for i, n in enumerate(NAMES):
    tp_rows, _, missed = matched_pairs(R["base"][n], OPERATING_THR)
    ious = np.array([r[3] for r in tp_rows]); ang = np.array([r[4] for r in tp_rows])
    quality[n] = {"IoU media": ious.mean(), "IoU mediana": np.median(ious),
                  "error ángulo medio (°)": ang.mean(), "error ángulo mediano (°)": np.median(ang),
                  "error ángulo p95 (°)": np.percentile(ang, 95)}
    hit = np.zeros(3); tot = np.zeros(3)
    for (img_i, j, *_ ) in tp_rows:
        hit[size_of(gt_rel[img_i][j])] += 1
    for img_i, rel in enumerate(gt_rel):
        for v in rel:
            tot[size_of(v)] += 1
    recall_by_size[n] = hit / np.maximum(tot, 1)
    pos = i - (len(NAMES) - 1) / 2
    axes[0].boxplot(ious, positions=[i], widths=0.5, patch_artist=True, showfliers=True,
                    boxprops=dict(facecolor=COLORS[n], alpha=0.8), medianprops=dict(color="#1a1a19"))
    axes[1].boxplot(ang, positions=[i], widths=0.5, patch_artist=True,
                    boxprops=dict(facecolor=COLORS[n], alpha=0.8), medianprops=dict(color="#1a1a19"))
    w = 0.8 / len(NAMES)
    b = axes[2].bar(np.arange(3) + pos * w, recall_by_size[n], w * 0.92, color=COLORS[n], label=n)
    axes[2].bar_label(b, fmt="%.2f", fontsize=7.5, padding=2, color="#52514e")
for ax in axes[:2]:
    ax.set_xticks(range(len(NAMES)), [n.split(" (")[0] for n in NAMES], fontsize=8)
    ax.grid(axis="x", visible=False)
axes[0].set(title="IoU de los aciertos", ylabel="IoU rotado")
axes[1].set(title="Error de ángulo de los aciertos", ylabel="grados")
axes[2].set(title=f"Recall por tamaño del billete (umbral {OPERATING_THR})", ylim=(0, 1.1))
axes[2].set_xticks(range(3), size_names, fontsize=8); axes[2].grid(axis="x", visible=False)
axes[2].legend(fontsize=7.5, loc="lower right")
plt.tight_layout(); plt.show()
display(pd.DataFrame(quality).T.style.format("{:.3f}"))
print("billetes por grupo de tamaño:", np.histogram(all_rel, edges)[0])

## 5. Estudio del NMS

Mismas predicciones, distinto umbral IoU del NMS. Un NMS muy estricto (0.1) elimina
billetes superpuestos (baja el recall); uno muy permisivo (0.7) deja cajas duplicadas
(baja la precisión).

In [ ]:
nms_tab = {(n, k): summarize(R["nms"][k][n], OPERATING_THR) for n in NAMES for k in NMS_STUDY}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for n in NAMES:
    for ax, m in zip(axes, ["mAP50-95", "precision", "recall"]):
        ax.plot(NMS_STUDY, [nms_tab[(n, k)][m] for k in NMS_STUDY], "o-", color=COLORS[n], label=n)
        ax.set(title=m, xlabel="NMS IoU")
for ax in axes:
    ax.axvline(NMS_IOU, color="#8a8983", lw=1)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()
display(pd.DataFrame({f"{n.split(' (')[0]} · {m}": [nms_tab[(n, k)][m] for k in NMS_STUDY]
                      for n in NAMES for m in ["f1", "mAP50-95"]}, index=pd.Index(NMS_STUDY, name="NMS IoU"))
        .style.format("{:.3f}"))

## 6. Estudio de rotación

Cada foto se gira (sentido antihorario, ampliando el lienzo con gris para no recortar nada)
y se rotan igual las anotaciones. Un detector OBB debería ser casi invariante; las caídas
señalan orientaciones poco vistas en el entrenamiento. 90° y 180° son giros exactos, sin
interpolación. Para ir más rápido, las fotos se reducen antes a 1600 px de lado largo (los
modelos trabajan a ≤ 1024 px, así que no cambia lo que ve la red).

In [ ]:
if RUN_STUDIES:
    rot = {(n, a): summarize(R["rot"][a][n], OPERATING_THR) for n in NAMES for a in ROT_ANGLES}
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    for n in NAMES:
        for ax, m in zip(axes, ["mAP50", "mAP50-95", "recall"]):
            ax.plot(ROT_ANGLES, [rot[(n, a)][m] for a in ROT_ANGLES], "o-", color=COLORS[n], label=n)
            ax.set(title=m if m != "recall" else f"recall (umbral {OPERATING_THR})", xlabel="giro (°)")
            ax.set_xticks(ROT_ANGLES)
    axes[0].legend(fontsize=8)
    plt.tight_layout(); plt.show()
    display(pd.DataFrame({f"{n.split(' (')[0]} · {m}": [rot[(n, a)][m] for a in ROT_ANGLES]
                          for n in NAMES for m in ["mAP50", "mAP50-95", "recall"]},
                         index=pd.Index(ROT_ANGLES, name="giro (°)")).style.format("{:.3f}"))

## 7. Estudio de resolución

La foto se reduce a distintos lados largos antes de dársela al modelo (simula una cámara de
peor resolución o un billete más lejos); el modelo la vuelve a escalar a su tamaño de
entrada. Por encima del tamaño de entrada del modelo (640 / 800) apenas debería cambiar.

**Nota**: el tamaño de entrada de la *red* va fijo en el ONNX (`--img-size` al exportar).
Para estudiar ese tamaño, exporta el mismo checkpoint a varios tamaños (p. ej. 512, 640,
800, 1024) en carpetas distintas de `models/`: aparecerán como modelos separados en todas
las tablas.

In [ ]:
if RUN_STUDIES:
    res = {(n, s): summarize(R["res"][s][n], OPERATING_THR) for n in NAMES for s in RES_SIZES}
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    for n in NAMES:
        for ax, m in zip(axes, ["mAP50", "mAP50-95", "recall"]):
            ax.plot(RES_SIZES, [res[(n, s)][m] for s in RES_SIZES], "o-", color=COLORS[n], label=n)
            ax.set(title=m if m != "recall" else f"recall (umbral {OPERATING_THR})", xlabel="lado largo de la foto (px)")
            ax.set_xscale("log"); ax.set_xticks(RES_SIZES, [str(s) for s in RES_SIZES]); ax.minorticks_off()
    axes[0].legend(fontsize=8)
    plt.tight_layout(); plt.show()
    display(pd.DataFrame({f"{n.split(' (')[0]} · {m}": [res[(n, s)][m] for s in RES_SIZES]
                          for n in NAMES for m in ["mAP50", "mAP50-95", "recall"]},
                         index=pd.Index(RES_SIZES, name="lado largo (px)")).style.format("{:.3f}"))

## 8. Velocidad y ejemplos de error

Latencia de `session.run` en CPU (ONNX Runtime, sin contar pre/post-proceso). Debajo, para
cada modelo, las fotos con más errores al umbral de trabajo: **anotación en blanco**,
**aciertos en verde**, **falsos positivos en rojo**; los billetes no detectados quedan solo
con el contorno blanco.

In [ ]:
lat = pd.DataFrame({n: {"media (ms)": np.mean(R["ms"][n]), "mediana (ms)": np.median(R["ms"][n]),
                        "p95 (ms)": np.percentile(R["ms"][n], 95)} for n in NAMES}).T
display(lat.style.format("{:.0f}"))

In [ ]:
N_EXAMPLES = 4

def draw_errors(img, r, thr):
    k = max(1, round(max(img.shape[:2]) / 700))
    tp, _, idx, _ = match(r, 0.5, thr)
    for poly in corners(r.gt).round().astype(np.int32):
        cv2.polylines(img, [poly], True, (255, 255, 255), 3 * k, cv2.LINE_AA)
    for d, ok in zip(idx, tp):
        poly = corners(r.boxes[d:d + 1])[0].round().astype(np.int32)
        c = (60, 175, 0) if ok else (40, 40, 230)
        cv2.polylines(img, [poly], True, c, 2 * k, cv2.LINE_AA)
        cv2.putText(img, f"{r.scores[d]:.2f}", tuple(int(v) for v in poly.min(0)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.9 * k, c, 2 * k, cv2.LINE_AA)
    return img

for n in NAMES:
    errs = []
    for i, r in enumerate(R["base"][n]):
        tp, _, _, g = match(r, 0.5, OPERATING_THR)
        errs.append(((~tp).sum() + g - tp.sum(), i))
    worst = [i for e, i in sorted(errs, reverse=True)[:N_EXAMPLES] if e > 0]
    if not worst:
        print(f"{n}: sin errores al umbral {OPERATING_THR}")
        continue
    fig, axes = plt.subplots(1, len(worst), figsize=(4 * len(worst), 4), squeeze=False)
    for ax, i in zip(axes[0], worst):
        img = draw_errors(load_image(samples[i].image_path), R["base"][n][i], OPERATING_THR)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.axis("off")
        tp, _, _, g = match(R["base"][n][i], 0.5, OPERATING_THR)
        ax.set_title(f"FP {int((~tp).sum())} · FN {int(g - tp.sum())}\n{samples[i].image_path.name[:28]}", fontsize=8)
    fig.suptitle(n, fontweight="bold", color=COLORS[n])
    plt.tight_layout(); plt.show()

## Resumen

Tabla final con las métricas del umbral común, el F1 con el umbral óptimo de cada modelo y
la robustez (peor mAP50 en el estudio de rotación y a 480 px).

In [ ]:
summary = main[["precision", "recall", "f1", "mAP50", "mAP75", "mAP50-95", "conf_mean", "conf_median", "conf_std", "ms/imagen"]].copy()
summary.insert(3, "F1 (umbral óptimo)", [opt.loc[n, "f1"] for n in NAMES])
summary.insert(4, "umbral óptimo", [best[n] for n in NAMES])
if RUN_STUDIES:
    summary["mAP50 peor giro"] = [min(rot[(n, a)]["mAP50"] for a in ROT_ANGLES) for n in NAMES]
    if 480 in RES_SIZES:
        summary["mAP50 @480px"] = [res[(n, 480)]["mAP50"] for n in NAMES]
cols = [c for c in summary.columns if c not in ("umbral óptimo", "ms/imagen", "conf_mean", "conf_median", "conf_std")]
display(summary.style.format({c: "{:.2f}" if c == "umbral óptimo" else "{:.0f}" if c == "ms/imagen" else "{:.3f}"
                              for c in summary.columns})
        .highlight_max(subset=cols, color="#cde2fb", axis=0))